In [ ]:
# =============================================================================
# MULTIMODAL NLP PIPELINE FOR GBS DETECTION (TEXT + STRUCTURED COVARIATES)
# =============================================================================
# Description:
# This script implements a fully reproducible multimodal NLP pipeline combining
# transformer-based text embeddings (CamemBERT-Bio) with structured covariates
# (age, sex, evolution) for automated detection of Guillain–Barré Syndrome (GBS).
#
# The pipeline evaluates five imbalance-handling strategies:
#   - Baseline
#   - Class-Weighted Cross-Entropy (CW)
#   - Focal Loss
#   - Class-Weighted Focal Loss (CW_FOCAL)
#   - Random Oversampling
#
# Key Features:
#   - Fixed train/test splits across 5 seeds
#   - Joint modeling of text + covariates
#   - Threshold tuning for optimal F1
#   - ROC–AUC, AUPRC, PPV/NPV, sensitivity/specificity
#   - Bootstrap confidence intervals for robust evaluation
#
# Requirements:
#   Python 3.11
#   transformers
#   datasets
#   torch
#   scikit-learn
#   pandas
#   numpy
#   scipy
#
# This script provides a research-grade multimodal framework for rare-event
# clinical NLP and enables direct comparison with the unimodal baseline.
# =============================================================================


**SCRIPT 1 — Generate Fixed Splits **

In [ ]:
# =============================================================================
# MAKE FIXED TRAIN/TEST SPLITS FOR MULTIMODAL PIPELINE
# =============================================================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Load multimodal dataset
df = pd.read_excel("GBS_FINAL_II.xlsx")
df = df.rename(columns={"Narratif": "text", "gbs_smq_label": "label"})
df["label"] = df["label"].astype(int)

# Covariates
df["age"] = df["Age_numeric"].astype(float)
df["sex"] = df["Sex_numeric"].astype(float)
df["evolution"] = df["Evolution_numeric"].astype(float)

# Normalize age
df["age"] = (df["age"] - df["age"].mean()) / df["age"].std()

# Save directory
split_dir = "/Users/keneni/Desktop/Bordeaux/Output/splits_multimodal"
os.makedirs(split_dir, exist_ok=True)

# Deterministic seeds
SPLIT_SEEDS = [42, 43, 44, 45, 46]
for seed in SPLIT_SEEDS:
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df["label"], random_state=seed
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=2/3, stratify=temp_df["label"], random_state=seed
    )
    train_df.to_csv(f"{splits_dir}/train_seed_{seed}.csv", index=False)
    val_df.to_csv(f"{splits_dir}/val_seed_{seed}.csv", index=False)
    test_df.to_csv(f"{splits_dir}/test_seed_{seed}.csv", index=False)
    print(f"Seed {seed}: Train={len(train_df)} (pos:{train_df['label'].sum()}), "
          f"Val={len(val_df)} (pos:{val_df['label'].sum()}), "
          f"Test={len(test_df)} (pos:{test_df['label'].sum()})")



**SCRIPT 2 — Full Sequential Multimodal Pipeline**

In [ ]:
# =============================================================================
# MULTIMODAL UNIFIED PIPELINE — SEQUENTIAL EXPERIMENTS
# FIXED SPLITS + PPV/NPV + BOOTSTRAP CI
# =============================================================================

import random
import numpy as np
import torch
import pandas as pd
import os
import warnings
import time
warnings.filterwarnings('ignore')

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModel,
    Trainer, TrainingArguments, set_seed, EarlyStoppingCallback
)

from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    confusion_matrix, roc_curve, auc, average_precision_score, f1_score
)
from scipy.special import softmax
import torch.nn as nn

# =============================================================================
# 0. CONFIGURATION
# =============================================================================

BASE_SEED = 42
SPLIT_SEEDS = [42, 43, 44, 45, 46]
N_BOOTSTRAP = 1000

EXPERIMENTS = ["BASELINE", "CW", "FOCAL", "CW_FOCAL", "OVERSAMPLING"]

split_dir = "/Users/keneni/Desktop/Bordeaux/Output/splits_multimodal"
output_root = "/Users/keneni/Desktop/Bordeaux/Output/Multimodal_Sequential"
os.makedirs(output_root, exist_ok=True)

device = torch.device("cpu")

# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

os.environ['PYTHONHASHSEED'] = '0'
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
torch.cuda.manual_seed_all(BASE_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_seed(BASE_SEED)

# =============================================================================
# 2. BOOTSTRAP CI
# =============================================================================

def bootstrap_ci(values, n_bootstrap=1000, seed=BASE_SEED):
    rng = np.random.RandomState(seed)
    values = np.array(values)
    boot = [np.mean(values[rng.choice(len(values), len(values), replace=True)])
            for _ in range(n_bootstrap)]
    return np.mean(values), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

# =============================================================================
# 3. LOSS FUNCTIONS
# =============================================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=[0.05, 0.95], gamma=2.0):
        super().__init__()
        self.alpha = torch.tensor(alpha, dtype=torch.float32)
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, labels):
        ce_loss = self.ce(logits, labels)
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha.to(logits.device)[labels]
        return (alpha_t * (1 - pt)**self.gamma * ce_loss).mean()

class FocalLossHybrid(nn.Module):
    def __init__(self, alpha, gamma, class_weights):
        super().__init__()
        self.alpha = torch.tensor(alpha, dtype=torch.float32)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, logits, labels):
        ce_loss = self.ce(logits, labels)
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha.to(logits.device)[labels]
        w_t = self.class_weights.to(logits.device)[labels]
        return (w_t * alpha_t * (1 - pt)**self.gamma * ce_loss).mean()

# =============================================================================
# 4. MULTIMODAL MODEL
# =============================================================================

class TransformerWithCovariates(nn.Module):
    def __init__(self, hf_name, n_covariates, loss_mode, class_weights=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(hf_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden + n_covariates, 2)

        self.loss_mode = loss_mode

        # ---- FIX: OVERSAMPLING uses CE (same as unimodal) ----
        if loss_mode in ["BASELINE", "OVERSAMPLING"]:
            self.ce = nn.CrossEntropyLoss()

        elif loss_mode == "CW":
            self.ce = nn.CrossEntropyLoss(weight=class_weights)

        elif loss_mode == "FOCAL":
            self.focal = FocalLoss()

        elif loss_mode == "CW_FOCAL":
            self.focal = FocalLossHybrid(alpha=[0.05, 0.95], gamma=2.0,
                                         class_weights=class_weights)

    def forward(self, input_ids, attention_mask, covariates, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        fused = torch.cat([cls, covariates.float()], dim=1)
        logits = self.classifier(fused)

        if labels is None:
            return {"logits": logits}

        # ---- FIX: OVERSAMPLING uses CE ----
        if self.loss_mode in ["BASELINE", "CW", "OVERSAMPLING"]:
            loss = self.ce(logits, labels)
        else:
            loss = self.focal(logits, labels)

        return {"loss": loss, "logits": logits}

# =============================================================================
# 5. TRAINING LOOP
# =============================================================================

def run_experiment(EXPERIMENT):

    print(f"\n\n==============================")
    print(f" RUNNING EXPERIMENT: {EXPERIMENT}")
    print(f"==============================\n")

    hf_name = "almanach/camembert-bio-base"
    tokenizer = AutoTokenizer.from_pretrained(hf_name)

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

    def add_covariates(example):
        example["covariates"] = [example["age"], example["sex"], example["evolution"]]
        return example

    all_results = []

    for run_seed in SPLIT_SEEDS:

        print(f"\n===== RUN WITH SEED {run_seed} | MODE = {EXPERIMENT} =====")
        start_time = time.time()
        set_seed(run_seed)

        # ---- Load fixed split ---
        train_df = pd.read_csv(f"{splits_dir}/train_seed_{run_seed}.csv")
        val_df = pd.read_csv(f"{splits_dir}/val_seed_{run_seed}.csv")
        test_df = pd.read_csv(f"{splits_dir}/test_seed_{run_seed}.csv")


        # ---- Oversampling ----
        if EXPERIMENT == "OVERSAMPLING":
            pos = train_df[train_df.label == 1]
            neg = train_df[train_df.label == 0]
            pos_os = pos.sample(len(neg), replace=True, random_state=run_seed)
            train_df = pd.concat([neg, pos_os]).sample(frac=1, random_state=run_seed)

        # ---- Class weights ----
        pos = train_df["label"].sum()
        neg = len(train_df) - pos
        class_weights = torch.tensor([1.0, neg / pos], dtype=torch.float32)

        # ---- Dataset ----
        train_ds = Dataset.from_pandas(train_df)
        val_ds = Dataset.from_pandas(val_df)

        test_ds = Dataset.from_pandas(test_df)

        train_ds = train_ds.map(tokenize, batched=True)
        val_ds = val_ds.map(tokenize, batched=True)

        test_ds = test_ds.map(tokenize, batched=True)

        train_ds = train_ds.map(add_covariates)
        val_ds = val_ds.map(add_covariates)

        test_ds = test_ds.map(add_covariates)

        train_ds.set_format(type="torch",
                            columns=["input_ids", "attention_mask", "covariates", "label"])
        val_ds.set_format(type="torch",
                            columns=["input_ids", "attention_mask", "covariates", "label"])
        test_ds.set_format(type="torch",
                           columns=["input_ids", "attention_mask", "covariates", "label"])

        # ---- Model init ----
        def model_init():
            return TransformerWithCovariates(
                hf_name=hf_name,
                n_covariates=3,
                loss_mode=EXPERIMENT,
                class_weights=class_weights
            )

        # ---- Training args ----
        args = TrainingArguments(
            output_dir=f"{output_root}/{EXPERIMENT}/seed_{run_seed}",
            learning_rate=2e-5,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            num_train_epochs=8,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            report_to="none",
            seed=run_seed
        )

        # ---- Metrics ----
        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=1)
            precision, recall, f1, _ = precision_recall_fscore_support(
                labels, preds, average="binary", zero_division=0
            )
            acc = accuracy_score(labels, preds)
            return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

        trainer = Trainer(
            model_init=model_init,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        trainer.train()
        best_epoch = trainer.state.best_epoch
        print(f"  Best epoch: {best_epoch} (early stopping)")
        # =============================================================
        # ---- TUNE THRESHOLD ON VALIDATION SET ----
        # =============================================================
        val_preds = trainer.predict(val_ds)
        val_probs = softmax(val_preds.predictions, axis=1)[:, 1]
        val_labels = val_preds.label_ids

        thresholds = np.linspace(0, 1, 101)
        val_f1s = [f1_score(val_labels, (val_probs >= th).astype(int)) for th in thresholds]
        best_th = thresholds[np.argmax(val_f1s)]
        best_val_f1 = max(val_f1s)

        print(f"  Optimal threshold (from val): {best_th:.3f} (F1: {best_val_f1:.4f})")
        # =============================================================
        # ---- FINAL EVALUATION ON TEST SET  ----
        # =============================================================
        test_preds = trainer.predict(test_ds)
        test_probs = softmax(test_preds.predictions, axis=1)[:, 1]
        test_labels = test_preds.label_ids


        # Apply threshold tuned on validation
        pred_labels = (test_probs >= best_th).astype(int)

        # ---- Confusion matrix ----
        cm = confusion_matrix(test_labels, pred_labels)
        tn, fp, fn, tp = cm.ravel()

        # ---- PPV & NPV ----
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0

        # ---- ROC / PR ----
        fpr, tpr, _ = roc_curve(test_labels, test_probs)
        roc_auc = auc(fpr, tpr)
        auprc = average_precision_score(test_labels, test_probs)

        # ---- Other metrics ----
        accuracy = accuracy_score(test_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            test_labels, pred_labels, average="binary", zero_division=0
        )

        sensitivity = tp/(tp+fn) if (tp+fn)>0 else 0
        specificity = tn/(tn+fp) if (tn+fp)>0 else 0

        runtime = time.time() - start_time

        all_results.append({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "ppv": ppv,
            "npv": npv,
            "roc_auc": roc_auc,
            "auprc": auprc,
            "threshold": best_th,
            "runtime_sec": runtime
        })

    return all_results

# =============================================================================
# 6. SEQUENTIAL EXECUTION OF ALL EXPERIMENTS
# =============================================================================

for EXPERIMENT in EXPERIMENTS:

    results = run_experiment(EXPERIMENT)

    print(f"\n===== FINAL RESULTS ({EXPERIMENT}) =====\n")

    metrics = [
        "accuracy","precision","recall","f1",
        "sensitivity","specificity","ppv","npv",
        "roc_auc","auprc","threshold","runtime_sec"
    ]

    for metric in metrics:
        values = [r[metric] for r in results]
        mean_val, low, high = bootstrap_ci(values, N_BOOTSTRAP)
        print(f"{metric:12}: {mean_val:.4f} [{low:.4f}-{high:.4f}]")
